# __Inteligencia Artificial - IUA 2026__

### *Pipeline de entrenamiento de modelos*

### Integrantes
* **Celeste Antonella Badami** — cbadami845@alumnos.iua.edu.ar
* **Benjamin Vargas** — bvargas161@alumnos.iua.edu.ar

---

### 1. Descripción del entrenamiento

El objetivo de este pipeline es entrenar un modelo de aprendizaje supervisado capaz de predecir el **tiempo estimado de arribo (`eta_segundos`)** de un vehículo de transporte público hacia una parada destino, a partir de variables telemétricas, espaciales, temporales, de estado del tráfico y condiciones meteorológicas.

#### Justificación de la elección del algoritmo (`LightGBM`)
Se seleccionó **LightGBM (`LGBMRegressor`)**, un algoritmo basado en *Gradient Boosted Decision Trees* (ensamble secuencial de árboles de decisión), por las siguientes razones técnicas y operativas:
1. **Modelado de relaciones no lineales:** El tiempo de viaje urbano depende de interacciones complejas entre distancia, velocidad instantánea, franja horaria, sentido de circulación y clima que los modelos lineales no logran capturar.
2. **Optimización directa de la métrica de negocio:** Permite entrenar utilizando la función de pérdida L1 (`regression_l1`), minimizando directamente el Error Absoluto Medio (MAE) y evitando que eventos atípicos extremos (embotellamientos severos) sesguen las predicciones habituales.
3. **Eficiencia computacional:** Mediante el uso de histogramas y crecimiento de árboles *leaf-wise*, entrena sobre más de $100{,}000$ registros en pocos segundos y realiza inferencias en milisegundos, requisito indispensable para sistemas de transporte en tiempo real.
4. **Soporte nativo para variables categóricas y valores nulos/cero:** Maneja etiquetas discretas sin necesidad de expandir la dimensionalidad mediante *One-Hot Encoding* y procesa de forma natural los registros donde el vehículo se encuentra detenido ($\text{velocidad} = 0$).

### 2. Preparación del entorno

In [20]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from datetime import datetime

In [9]:
# Carga de dataset de entrenamiento

df = pd.read_csv('../data/eda_dataset.csv')

### 3. Entrenamiento de modelo

#### 3.1 Ingenieria de features

Se añaden/eliminan algunas features dadas las caracterisitcas del dataset de entrenamiento

In [10]:
df = df.drop(columns=['es_fin_de_semana'])  # Columna constante (todo ceros)

# Tiempos teóricos ideales y actuales según física básica (t = d / v)
df['eta_teorico_freeflow'] = (df['distancia_restante_km'] / df['free_flow_speed']) * 3600
df['eta_teorico_actual'] = (df['distancia_restante_km'] / df['current_speed']) * 3600

# Ratios de congestión y progreso del recorrido
df['ratio_velocidad'] = df['current_speed'] / df['free_flow_speed']
df['ratio_tiempo_via'] = df['current_travel_time'] / df['freeflow_travel_time']
df['progreso_recorrido'] = df['distancia_en_traza'] / (
    df['distancia_en_traza'] + df['distancia_restante_km']
)

#### 3.2 Declaración de variable categoricas

Se hace para avisarle a LightGBM que esos números son etiquetas de grupo (como un código de clima o un día de la semana) y no cantidades matemáticas reales.

In [11]:
cat_cols = [
    'direction_id',
    'weather_code',
    'current_status_encoded',
    'dia_semana',
    'occupancy_encoded',
    'is_day_encoded'
]
for col in cat_cols:
    df[col] = df[col].astype('category')

#### 3.3 División de set de entrenamiento

__(70% Train / 10% Val / 20% Test)__

No se usa shuffle=True para evitar que paradas del mismo viaje queden en Train y Test

In [12]:
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.80)

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

X_train, y_train = train.drop(columns=['eta_segundos']), train['eta_segundos']
X_val, y_val = val.drop(columns=['eta_segundos']), val['eta_segundos']
X_test, y_test = test.drop(columns=['eta_segundos']), test['eta_segundos']

#### 3.4 Entrenamiento del modelo

In [13]:
model = lgb.LGBMRegressor(
    objective='regression_l1',  # Optimiza directamente el MAE (robusto ante outliers)
    metric='mae',
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=31,
    subsample_for_bin=50000,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    min_child_samples=30,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    categorical_feature=cat_cols,
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

/home/ben/Documentos/IA/proyecto-final/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3025
[LightGBM] [Info] Number of data points in the train set: 74781, number of used features: 23
[LightGBM] [Info] Start training from score 850.512451
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 169.6
[200]	valid_0's l1: 152.66
[300]	valid_0's l1: 149.33
[400]	valid_0's l1: 147.42
Early stopping, best iteration is:
[408]	valid_0's l1: 147.281


,learning_rate,0.03
,n_estimators,2000
,subsample_for_bin,50000
,objective,'regression_l1'
,min_child_samples,30
,subsample,0.8
,subsample_freq,1
,colsample_bytree,0.8
,random_state,42
,n_jobs,-1
,metric,'mae'


### 4. Metricas de modelo

In [ ]:
preds = model.predict(X_test)
# Aseguramos que ninguna predicción sea menor a 0 por seguridad operativa
preds = np.clip(preds, a_min=0, a_max=None)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
r2 = r2_score(y_test, preds)

print("\n=== MÉTRICAS GLOBALES EN TEST ===")
print(f"MAE  : {mae:.2f} segundos ({mae/60:.2f} minutos)")
print(f"RMSE : {rmse:.2f} segundos ({rmse/60:.2f} minutos)")
print(f"MAPE : {mape:.2f}%")
print(f"R²   : {r2:.4f}")


=== MÉTRICAS GLOBALES EN TEST ===
MAE  : 153.11 segundos (2.55 minutos)
RMSE : 224.97 segundos (3.75 minutos)
MAPE : 19.79%
R²   : 0.9045


In [ ]:
# Analisis de error por tramo
test_eval = test.copy()
test_eval['pred'] = preds
test_eval['error_abs_seg'] = np.abs(test_eval['eta_segundos'] - test_eval['pred'])
test_eval['error_pct'] = (test_eval['error_abs_seg'] / test_eval['eta_segundos']) * 100
test_eval['tramo'] = pd.cut(
    test_eval['distancia_restante_km'],
    bins=[0, 1, 3, 10],
    labels=['< 1 km (Cerca)', '1 - 3 km (Medio)', '> 3 km (Lejos)']
)

print("\n=== RENDIMIENTO POR TRAMO DE DISTANCIA ===")
print(test_eval.groupby('tramo', observed=False)[['error_abs_seg', 'error_pct']].mean().round(2))


=== RENDIMIENTO POR TRAMO DE DISTANCIA ===
                  error_abs_seg  error_pct
tramo                                     
< 1 km (Cerca)            85.42      30.05
1 - 3 km (Medio)         135.84      18.84
> 3 km (Lejos)           213.39      14.27


In [ ]:
# Importancia de variables
importancias = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n=== TOP 10 VARIABLES MÁS IMPORTANTES ===")
print(importancias.head(10))


=== TOP 10 VARIABLES MÁS IMPORTANTES ===
hora_sin                 1646
distancia_en_traza       1601
hora_cos                 1402
distancia_restante_km    1111
temperature_2m            898
current_travel_time       650
relative_humidity_2m      579
velocidad_traza_kmh       486
direction_id              455
progreso_recorrido        430
dtype: int32


### 5. Respaldo de modelos

In [22]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_archivo = f"../models/modelo_eta_lgbm_{timestamp}_mae{mae:.0f}s.pkl"
joblib.dump(model, nombre_archivo)
print(f"\nModelo guardado exitosamente como '{nombre_archivo}'")


Modelo guardado exitosamente como '../models/modelo_eta_lgbm_20261112_121344_mae153s.pkl'
